# 02 — Preprocessing

This notebook prepares the predictive-maintenance dataset for the two modeling stages.

**Model 1 target:** `Machine failure`

**Model 2 targets:** `TWF`, `HDF`, `PWF`, `OSF` (multi-label failure types).

The test set is kept separate for final evaluation.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

DATA_PATH = "../data/predictive_maintenance.csv"
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())


Shape: (10000, 14)
Columns: ['UDI', 'Product ID', 'Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']


## 1. Failure-type audit

The source data contains five failure-type flags. `RNF` occurs only once, so it is kept for reference but not trained as a separate label in this version. Because some failed rows contain more than one type, Model 2 is multi-label.

In [2]:
failure_types_all = ["TWF", "HDF", "PWF", "OSF", "RNF"]
failures = df[df["Machine failure"] == 1].copy()

print("Total failures:", len(failures))
print("\nFailure type counts:")
print(failures[failure_types_all].sum())
print("\nRows with >1 failure type:", (failures[failure_types_all].sum(axis=1) > 1).sum())


Total failures: 339

Failure type counts:
TWF     46
HDF    115
PWF     95
OSF     98
RNF      1
dtype: int64

Rows with >1 failure type: 24


## 2. Model 1 split

Inputs use machine type plus five sensor readings. Identifier columns and direct failure-type flags are not inputs to Model 1.

In [3]:
feature_cols = [
    "Type",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

X = df[feature_cols].copy()
y = df["Machine failure"].copy()

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.20, stratify=y_temp, random_state=42
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)
print("\nTrain labels:\n", y_train.value_counts())
print("\nValidation labels:\n", y_val.value_counts())
print("\nTest labels:\n", y_test.value_counts())


Train: (6400, 6)
Validation: (1600, 6)
Test: (2000, 6)

Train labels:
 Machine failure
0    6183
1     217
Name: count, dtype: int64

Validation labels:
 Machine failure
0    1546
1      54
Name: count, dtype: int64

Test labels:
 Machine failure
0    1932
1      68
Name: count, dtype: int64


In [4]:
categorical_features = ["Type"]
numeric_features = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ("num", StandardScaler(), numeric_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Processed:", X_train_processed.shape, X_val_processed.shape, X_test_processed.shape)


Processed: (6400, 8) (1600, 8) (2000, 8)


## 3. Model 2 failure-only split

We keep only rows where `Machine failure == 1` and predict the four learnable failure-type labels. Multi-label stratification is used because several rows contain multiple failure types.

In [5]:
failure_labels = ["TWF", "HDF", "PWF", "OSF"]
failure_df = df[df["Machine failure"] == 1].copy()
X_failure = failure_df[feature_cols].copy()
Y_failure = failure_df[failure_labels].copy()

split_1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_val_idx, test_idx = next(split_1.split(X_failure, Y_failure))

X_failure_train_val = X_failure.iloc[train_val_idx].copy()
Y_failure_train_val = Y_failure.iloc[train_val_idx].copy()
X_failure_test = X_failure.iloc[test_idx].copy()
Y_failure_test = Y_failure.iloc[test_idx].copy()

split_2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, val_idx = next(split_2.split(X_failure_train_val, Y_failure_train_val))

X_failure_train = X_failure_train_val.iloc[train_idx].copy()
Y_failure_train = Y_failure_train_val.iloc[train_idx].copy()
X_failure_val = X_failure_train_val.iloc[val_idx].copy()
Y_failure_val = Y_failure_train_val.iloc[val_idx].copy()

print("Failure train/val/test:", X_failure_train.shape, X_failure_val.shape, X_failure_test.shape)
print("\nTrain label counts:\n", Y_failure_train.sum())
print("\nValidation label counts:\n", Y_failure_val.sum())
print("\nTest label counts:\n", Y_failure_test.sum())


Failure train/val/test: (216, 6) (55, 6) (68, 6)

Train label counts:
 TWF    29
HDF    73
PWF    61
OSF    62
dtype: int64

Validation label counts:
 TWF     8
HDF    19
PWF    15
OSF    16
dtype: int64

Test label counts:
 TWF     9
HDF    23
PWF    19
OSF    20
dtype: int64


In [6]:
failure_preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ("num", StandardScaler(), numeric_features)
])

X_failure_train_processed = failure_preprocessor.fit_transform(X_failure_train)
X_failure_val_processed = failure_preprocessor.transform(X_failure_val)
X_failure_test_processed = failure_preprocessor.transform(X_failure_test)

print("Model 2 processed:", X_failure_train_processed.shape, X_failure_val_processed.shape, X_failure_test_processed.shape)


Model 2 processed: (216, 8) (55, 8) (68, 8)


## Output

The objects created here are consumed by the model notebooks. Keeping preprocessing separate prevents accidental fitting on the test set.